# 00 — Сборка датасета

Объединяем три русскоязычных датасета по эмоциям в один большой:

| Датасет | Размер | Источник | Классы |
|---|---|---|---|
| `seara/ru_go_emotions` | ~54k | Reddit (пер. Google Translate) | 28 эмоций → Ekman |
| `cedr` | ~9.4k | Блоги, новости, Twitter (нативный RU) | 5 эмоций → Ekman |
| `Djacon/ru-izard-emotions` | ~30k | Reddit (пер. DeepL) | 10 эмоций → Ekman |

**Все приводятся к единой Ekman-таксономии (7 классов):**
`anger`, `disgust`, `fear`, `joy`, `sadness`, `surprise`, `neutral`

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(WORKING_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
print('Setup done')

## 1. Загружаем каждый датасет отдельно

In [ ]:
from src.data_loader import (
    load_ru_go_emotions, load_cedr, load_ru_izard_emotions,
    merge_datasets, EKMAN_ID2LABEL, EKMAN_LABEL_NAMES,
)

print('=== seara/ru_go_emotions ===')
ds_goe = load_ru_go_emotions(config='simplified')

print('\n=== cedr ===')
ds_cedr = load_cedr()

print('\n=== Djacon/ru-izard-emotions ===')
ds_izard = load_ru_izard_emotions()

In [ ]:
# Смотрим распределение эмоций в каждом датасете отдельно
EMOTION_COLORS = {
    'anger': '#e74c3c', 'disgust': '#8e44ad', 'fear': '#2c3e50',
    'joy': '#f39c12', 'sadness': '#3498db', 'surprise': '#1abc9c', 'neutral': '#95a5a6',
}

datasets_raw = {
    'ru_go_emotions': ds_goe,
    'cedr':           ds_cedr,
    'ru_izard':       ds_izard,
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, (name, ds) in zip(axes, datasets_raw.items()):
    df = pd.concat([ds[s].to_pandas() for s in ds])
    counts = df['label'].map(EKMAN_ID2LABEL).value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0)
    colors = [EMOTION_COLORS[e] for e in counts.index]
    ax.bar(counts.index, counts.values, color=colors)
    ax.set_title(f'{name}\n(всего {len(df):,})')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=40)
    for i, v in enumerate(counts.values):
        if v > 0:
            ax.text(i, v + 10, f'{int(v):,}', ha='center', fontsize=7)

plt.suptitle('Распределение эмоций в каждом датасете', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/per_dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Объединяем в один датасет

In [ ]:
dataset = merge_datasets(
    datasets_raw,
    test_size=0.15,
    val_size=0.15,
    seed=42,
)

In [ ]:
# Анализ итогового датасета
all_df = pd.concat([dataset[s].to_pandas() for s in dataset])
all_df['emotion'] = all_df['label'].map(EKMAN_ID2LABEL)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Итоговое распределение
counts = all_df['emotion'].value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0)
colors = [EMOTION_COLORS[e] for e in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0].set_title(f'Объединённый датасет (всего {len(all_df):,})')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(v):,}\n({v/len(all_df)*100:.1f}%)', ha='center', fontsize=8)

# По сплитам
split_data = {}
for split in ['train', 'validation', 'test']:
    df = dataset[split].to_pandas()
    split_data[split] = df['label'].map(EKMAN_ID2LABEL).value_counts(normalize=True).reindex(EKMAN_LABEL_NAMES).fillna(0)
pd.DataFrame(split_data).plot(kind='bar', ax=axes[1], color=['#3498db','#e67e22','#9b59b6'])
axes[1].set_title('Распределение по сплитам (доли)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/merged_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Проверка дисбаланса классов
print('Дисбаланс классов (объединённый датасет):')
print(f'{"Класс":<12} {"Кол-во":>8} {"Доля":>8} {"Ratio к min":>12}')
counts_series = all_df['emotion'].value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0)
min_cnt = counts_series.min()
for emotion, cnt in counts_series.items():
    ratio = cnt / min_cnt
    flag = '  ⚠ сильный дисбаланс' if ratio > 5 else ''
    print(f'{emotion:<12} {int(cnt):>8,} {cnt/len(all_df)*100:>7.1f}% {ratio:>10.1f}x{flag}')

## 3. Обработка дисбаланса (опционально)

Если `joy` или `neutral` сильно преобладают, можно:
- **Upsampling** редких классов (рекомендуется)
- **Downsampling** частых классов
- **class_weight** при обучении

Ниже — мягкий downsampling: ограничиваем максимальный размер класса.

In [ ]:
from datasets import Dataset
from sklearn.utils import resample

MAX_PER_CLASS = 10_000  # Ограничение на класс (можно убрать или изменить)

train_df = dataset['train'].to_pandas()

balanced_parts = []
for label_id in range(len(EKMAN_LABEL_NAMES)):
    subset = train_df[train_df['label'] == label_id]
    if len(subset) == 0:
        print(f'WARNING: нет примеров для класса {EKMAN_ID2LABEL[label_id]}')
        continue
    if len(subset) > MAX_PER_CLASS:
        subset = resample(subset, n_samples=MAX_PER_CLASS, random_state=42, replace=False)
    balanced_parts.append(subset)

balanced_train_df = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Train до балансировки:  {len(train_df):,}')
print(f'Train после балансировки: {len(balanced_train_df):,}')
print('\nРаспределение после:')
print(balanced_train_df['label'].map(EKMAN_ID2LABEL).value_counts().reindex(EKMAN_LABEL_NAMES).fillna(0).astype(int))

In [ ]:
from datasets import DatasetDict

# Собираем финальный датасет
dataset_final = DatasetDict({
    'train':      Dataset.from_pandas(balanced_train_df),
    'validation': dataset['validation'],
    'test':       dataset['test'],
})
print('Финальный датасет:')
for split in dataset_final:
    print(f'  {split:12s}: {len(dataset_final[split]):,}')

## 4. Сохранение

In [ ]:
from src.preprocessor import preprocess_batch

# Очищаем тексты
print('Очищаю тексты...')
dataset_clean = dataset_final.map(
    lambda batch: preprocess_batch(batch, text_column='text', clean=True),
    batched=True, batch_size=1000, desc='Cleaning',
)
print('Готово!')

In [ ]:
# Сохраняем
DATA_SAVE_PATH = f'{WORKING_DIR}/processed_data'
dataset_clean.save_to_disk(DATA_SAVE_PATH)

with open(f'{WORKING_DIR}/label_names.json', 'w') as f:
    json.dump(EKMAN_LABEL_NAMES, f)

for split in ['train', 'validation', 'test']:
    df = dataset_clean[split].to_pandas()
    if 'label' in df.columns:
        df['emotion'] = df['label'].map(EKMAN_ID2LABEL)
    df.to_csv(f'{WORKING_DIR}/{split}.csv', index=False, encoding='utf-8')
    print(f'  {split}.csv: {len(df):,} строк')

print(f'\nВсё сохранено в: {WORKING_DIR}')
print(f'Следующий шаг: запустить 01_eda_and_preprocessing.ipynb')